# Demo: final pipeline on an unseen SoccerNet clip

This notebook runs the **final** pipeline (fine-tuned YOLO11l + BoT-SORT + fine-tuned OSNet ReID + offline post-processing) on a SoccerNet Tracking clip that was **never used for training or validation**, and renders a side-by-side video (ground truth | predictions) plus a short GIF.

**Runtime:** select *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The SoccerNet test split download takes a few minutes.

In [ ]:
# 1. Code and dependencies (separate installs, so a failure is easy to spot)
import sys; print("Python", sys.version.split()[0])

!git clone -q https://github.com/RaffaeleRosanova01/soccer-player-tracking.git
%cd soccer-player-tracking

!pip install -q SoccerNet "ultralytics==8.3.246"
# BoxMOT 16.0.7 (the version used for the contest) declares Python < 3.13 and pins some old
# dependencies: install it without the version check and add its runtime dependencies.
!pip install -q filterpy gdown lapx loguru ftfy yacs scikit-learn
!pip install -q --no-deps --ignore-requires-python "boxmot @ git+https://github.com/mikel-brostrom/boxmot.git@v16.0.7"

# Sanity check: all the libraries used below must import
import SoccerNet, ultralytics, boxmot
print("OK -> ultralytics", ultralytics.__version__, "| boxmot", boxmot.__version__)


In [ ]:
# 2. Fine-tuned weights (from the GitHub release)
import os, urllib.request

RELEASE_URL = "https://github.com/RaffaeleRosanova01/soccer-player-tracking/releases/download/v1.0/"
WEIGHTS = ["yolov11l_finetuned_best.pt", "osnet_x1_0_finetuned_best.pt"]

os.makedirs("weights", exist_ok=True)
for name in WEIGHTS:
    path = os.path.join("weights", name)
    if not os.path.exists(path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(RELEASE_URL + name, path)
print("Weights ready:", os.listdir("weights"))

In [ ]:
# 3. One held-out clip from the SoccerNet Tracking 2023 test split
import json, zipfile
from SoccerNet.Downloader import SoccerNetDownloader

SoccerNetDownloader(LocalDirectory="soccernet").downloadDataTask(task="tracking-2023", split=["test"])
archive = zipfile.ZipFile("soccernet/tracking-2023/test.zip")

# Same split as the main notebook: the first 30 test sequences were used for validation,
# the remaining ones were never seen during training or tuning.
sequences = sorted({part for n in archive.namelist() for part in n.split("/") if part.startswith("SNMOT-")})
held_out = sequences[30:]
print("Held-out sequences:", held_out)

SEQ_INDEX = 0                      # change to render a different clip
SEQ = held_out[SEQ_INDEX]
seq_dir = os.path.join("data", "demo", SEQ)

for member in archive.namelist():
    if f"/{SEQ}/" in "/" + member and not member.endswith("/"):
        rel = member[member.index(SEQ):]
        os.makedirs(os.path.dirname(os.path.join("data", "demo", rel)), exist_ok=True)
        with archive.open(member) as src, open(os.path.join("data", "demo", rel), "wb") as dst:
            dst.write(src.read())

# Contest ROIs (normalized coordinates)
rois = {"roi1": {"x": 0.01, "y": 0.01, "width": 0.4, "height": 0.75},
        "roi2": {"x": 0.5, "y": 0.35, "width": 0.5, "height": 0.5}}
with open(os.path.join(seq_dir, "roi.json"), "w") as f:
    json.dump(rois, f, indent=2)
print(SEQ, "->", len(os.listdir(os.path.join(seq_dir, "img1"))), "frames")

In [ ]:
# 4. Run the full pipeline (detection -> tracking -> ROI counting -> post-processing)
!python run_pipeline.py --data-dir data/demo --output-dir predictions --weights-dir weights

In [ ]:
# 5. Side-by-side video + README GIF
!python tools/make_comparison_video.py --seq-dir {seq_dir} --pred-file predictions/tracking_{SEQ}_23.txt --out demo_{SEQ}.mp4 --gif demo.gif --gif-start 0 --gif-seconds 6

from IPython.display import Image, display
display(Image("demo.gif"))

In [ ]:
# 6. Download the results
from google.colab import files
files.download("demo.gif")
files.download(f"demo_{SEQ}.mp4")